# 2. The turn: steps, priority, and the stack

A turn is twelve steps. Between them, players get *priority* — the right to act.
Almost every timing rule in Magic is a statement about priority, so this notebook
is mostly about who is allowed to do what, when.

The engine's core safety property lives here: **it enumerates the legal actions
and asks an agent to pick one.** An agent therefore *cannot* make an illegal
play, and any rules violation is an engine bug rather than a silent cheat.

In [1]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from manalab.cards.mana import parse_cost
from manalab.cards.model import Card
from manalab.cards.store import CardStore

store = CardStore(REPO / "data" / "cards.duckdb")


def creature(name, power, toughness, *keywords, cost="{1}{G}"):
    # A synthetic creature, so a demo shows a rule and not a card.
    return Card(name.lower().replace(" ", "-"), name, parse_cost(cost),
                "Creature - Test", power=str(power), toughness=str(toughness),
                keywords=frozenset(keywords))


FOREST = store.by_name("Forest")

In [2]:
from manalab.engine.zones import Zone


def show(state, title=""):
    # Print the whole visible game state. The workhorse of these notebooks.
    if title:
        print(f"--- {title}")
    print(f"    turn {state.turn}, active P{state.active_player}")
    for player in state.players:
        flags = " LOST" if player.has_lost else ""
        print(f"    P{player.index}: {player.life} life{flags}"
              f"   hand={len(state.hand(player.index))}"
              f"  library={len(state.library(player.index))}"
              f"  graveyard={len(state.graveyard(player.index))}")
        for obj in state.battlefield(player.index):
            marks = []
            if obj.tapped:
                marks.append("tapped")
            if obj.damage:
                marks.append(f"dmg{obj.damage}")
            if obj.deathtouched:
                marks.append("deathtouched")
            if obj.summoning_sick:
                marks.append("sick")
            if obj.attacking is not None:
                marks.append(f"attacking P{obj.attacking}")
            if obj.blocking is not None:
                marks.append("blocking")
            if obj.counters:
                marks.extend(f"{k}x{v}" for k, v in sorted(obj.counters.items()))
            body = f"{obj.power}/{obj.toughness}" if obj.is_creature else "land"
            suffix = f"  [{' '.join(marks)}]" if marks else ""
            print(f"        #{obj.id} {obj.card.name:<22} {body:<6}{suffix}")
    stack = state.zone(Zone.STACK)
    if stack:
        print(f"    stack (top last): {[o.card.name for o in stack]}")

## The twelve steps

In [3]:
from manalab.engine.steps import MAIN_STEPS, NO_PRIORITY, PHASE_OF, TURN

for i, step in enumerate(TURN, 1):
    tags = []
    if step in NO_PRIORITY:
        tags.append("no priority")
    if step in MAIN_STEPS:
        tags.append("sorcery speed")
    note = f"   ({', '.join(tags)})" if tags else ""
    print(f"  {i:>2}. {step.value:<22} {PHASE_OF[step].value}{note}")

   1. untap                  beginning   (no priority)
   2. upkeep                 beginning
   3. draw                   beginning
   4. precombat main         precombat main   (sorcery speed)
   5. beginning of combat    combat
   6. declare attackers      combat
   7. declare blockers       combat
   8. combat damage          combat
   9. end of combat          combat
  10. postcombat main        postcombat main   (sorcery speed)
  11. end                    ending
  12. cleanup                ending   (no priority)


Two steps are special and both are classic bug sources:

- **Untap** — no player gets priority (CR 502.4). Nothing can be cast.
- **Cleanup** — no priority either, unless something triggers.

## Sorcery speed has three conditions, and all three are load-bearing

You may cast a sorcery only when it is **your** main phase, the stack is
**empty**, and you have priority. Drop the stack check and you can cast a sorcery
in response to something — the classic way this goes wrong.

In [4]:
from manalab.engine.steps import Step, is_sorcery_speed

print(f"{'step':<22}{'active?':<10}{'stack empty?':<15}sorcery speed?")
for step in (Step.PRECOMBAT_MAIN, Step.DECLARE_ATTACKERS, Step.UPKEEP):
    for active in (True, False):
        for empty in (True, False):
            ok = is_sorcery_speed(step, active, empty)
            print(f"  {step.value:<20}{str(active):<10}{str(empty):<15}{ok}")

step                  active?   stack empty?   sorcery speed?
  precombat main      True      True           True
  precombat main      True      False          False
  precombat main      False     True           False
  precombat main      False     False          False
  declare attackers   True      True           False
  declare attackers   True      False          False
  declare attackers   False     True           False
  declare attackers   False     False          False
  upkeep              True      True           False
  upkeep              True      False          False
  upkeep              False     True           False
  upkeep              False     False          False


## The action vocabulary

`engine.actions.legal()` returns everything a player may do right now. `Pass` is
always in the list — a player may always decline, and a priority loop that can
produce an empty list will deadlock.

In [5]:
from manalab.engine import actions as A
from manalab.engine.state import GameState

bear = creature("Bear", 2, 2, cost="{1}{G}")
state = GameState.new([[], []], seed=0)
state.active_player = 0
for _ in range(2):
    state.create(FOREST, owner=0, zone=Zone.BATTLEFIELD)
state.create(bear, owner=0, zone=Zone.HAND)
state.create(FOREST, owner=0, zone=Zone.HAND)

options = A.legal(state, 0, Step.PRECOMBAT_MAIN, stack_empty=True)
print("in my main phase, two Forests untapped, Bear + Forest in hand:")
for option in options:
    print("   ", option.describe(state))

in my main phase, two Forests untapped, Bear + Forest in hand:
    pass
    play Forest
    cast Bear


In [6]:
# The same board in the opponent's declare-attackers step: no sorcery speed.
options = A.legal(state, 0, Step.DECLARE_ATTACKERS, stack_empty=True)
print("during combat (no sorcery speed, Bear is not an instant):")
for option in options:
    print("   ", option.describe(state))

during combat (no sorcery speed, Bear is not an instant):
    pass


## Casting: hand → stack → battlefield

Casting is not "the creature appears". It is a move to the stack, then a
*separate* resolution step. Anything can happen in between — that gap is where
counterspells and removal-in-response live.

In [7]:
from manalab.engine.agents import CurveAgent, PassingAgent
from manalab.engine.game import Game

state = GameState.new([[], []], seed=0)
state.active_player = 0
for _ in range(2):
    state.create(FOREST, owner=0, zone=Zone.BATTLEFIELD)
spell = state.create(bear, owner=0, zone=Zone.HAND)

game = Game(state, [CurveAgent(), PassingAgent()])
show(state, "before casting")

game._apply(0, A.CastSpell(spell.id))
show(state, "after casting — it is on the stack, not the battlefield")

game._resolve_top()
show(state, "after resolving")

--- before casting
    turn 0, active P0
    P0: 20 life   hand=1  library=0  graveyard=0
        #1 Forest                 land  
        #2 Forest                 land  
    P1: 20 life   hand=0  library=0  graveyard=0
--- after casting — it is on the stack, not the battlefield
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=0
        #1 Forest                 land    [tapped]
        #2 Forest                 land    [tapped]
    P1: 20 life   hand=0  library=0  graveyard=0
    stack (top last): ['Bear']
--- after resolving
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=0
        #1 Forest                 land    [tapped]
        #2 Forest                 land    [tapped]
        #5 Bear                   2/2     [sick]
    P1: 20 life   hand=0  library=0  graveyard=0


Notice the Forests are now **tapped** — paying the cost happened on
announcement, not on resolution. And notice the creature on the battlefield has a
different id from the card in hand, for the CR 400.7 reason from notebook 1.

## State-based actions

Some things are not effects and never use the stack: a creature with lethal
damage dies, a player at 0 life loses. The game checks these *whenever a player
would receive priority*, and applies them repeatedly until none apply.

The repetition matters — one creature dying can put another below zero toughness.

In [8]:
from manalab.engine import sba

state = GameState.new([[], []], seed=0)
small = state.create(creature("Small", 1, 1), owner=0, zone=Zone.BATTLEFIELD)
big = state.create(creature("Big", 5, 5), owner=0, zone=Zone.BATTLEFIELD)

small.damage = 1          # lethal
big.damage = 2            # not lethal
state.players[1].life = 0 # dead

show(state, "before state-based actions")
events = sba.check(state)
print("\n  applied:", events)
show(state, "after")

--- before state-based actions
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=0
        #1 Small                  1/1     [dmg1]
        #2 Big                    5/5     [dmg2]
    P1: 0 life   hand=0  library=0  graveyard=0

  applied: ['P1 loses (life)', 'Small dies']
--- after
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=1
        #2 Big                    5/5     [dmg2]
    P1: 0 life LOST   hand=0  library=0  graveyard=0


## The priority loop, in full

Now put it together. `Game._priority` is the heart of the engine:

1. Check state-based actions.
2. Enumerate legal actions for whoever has priority.
3. Ask that agent to pick one; **raise if it picks something not in the list**.
4. If they acted, they get priority again (CR 117.3c).
5. If everyone passed in succession: resolve the top of the stack, or end the step.

Let us watch one full turn, step by step.

In [9]:
library = [FOREST] * 10 + [bear] * 10
state = GameState.new([list(library), list(library)], seed=3)
game = Game(state, [CurveAgent(), PassingAgent()], max_turns=4)
game.setup()

before = len(state.log)
game._play_turn()
print("turn 1, as the engine logged it:")
for line in state.log[before:]:
    print("   ", line)
show(state, "end of turn 1")

turn 1, as the engine logged it:
    --- turn 1 (P0)
    Forest: hand -> battlefield
--- end of turn 1
    turn 1, active P0
    P0: 20 life   hand=6  library=13  graveyard=0
        #55 Forest                 land    [sick]
    P1: 20 life   hand=7  library=13  graveyard=0


In [10]:
for _ in range(3):
    game._play_turn()
show(state, "end of turn 4")
print("\nfull log:")
for line in state.log:
    print("   ", line)

--- end of turn 4
    turn 4, active P1
    P0: 20 life   hand=5  library=12  graveyard=0
        #55 Forest                 land    [tapped]
        #58 Forest                 land    [tapped sick]
        #60 Bear                   2/2     [sick]
    P1: 20 life   hand=9  library=11  graveyard=0

full log:
    Forest: library -> hand
    Forest: library -> hand
    Forest: library -> hand
    Forest: library -> hand
    Bear: library -> hand
    Bear: library -> hand
    Forest: library -> hand
    Bear: library -> hand
    Bear: library -> hand
    Forest: library -> hand
    Forest: library -> hand
    Forest: library -> hand
    Bear: library -> hand
    Bear: library -> hand
    --- turn 1 (P0)
    Forest: hand -> battlefield
    --- turn 2 (P1)
    Bear: library -> hand
    --- turn 3 (P0)
    Bear: library -> hand
    Forest: hand -> battlefield
    Bear: hand -> stack
    Bear: stack -> battlefield
    --- turn 4 (P1)
    Forest: library -> hand


## An agent cannot cheat

The contract is worth demonstrating. An agent that returns an action it was not
offered gets an exception, not a silent illegal play.

In [11]:
class CheatingAgent:
    name = "cheater"

    def act(self, state, player, options):
        return A.PlayLand(999_999)  # never offered

# Libraries must be big enough to survive the opening draw — with five cards
# each, both players lose to drawing from an empty library before the agent is
# ever asked for an action, and this cell would quietly demonstrate nothing.
state = GameState.new([[FOREST] * 30, [FOREST] * 30], seed=0)
game = Game(state, [CheatingAgent(), PassingAgent()], max_turns=1)
try:
    game.run()
    print("no exception — the illegal action was accepted (this would be a bug)")
except ValueError as exc:
    print("ValueError:", exc)

ValueError: cheater chose an illegal action


## What to take into notebook 3

- Twelve steps; untap and cleanup grant no priority.
- Sorcery speed = your main phase **and** empty stack **and** priority.
- Casting is two events: onto the stack, then resolution. Costs are paid on the way up.
- State-based actions run to a fixed point whenever priority would be received.
- The engine enumerates legal actions; an agent that invents one raises.

Next: combat — the part with the most rules per line of code.